# TSForecasting: Baseline Pipeline

This notebook demonstrates the main TSForecasting pipeline with all available options for quick, high-level time series forecasting.

## Topics Covered

- Data generation with `TimeSeriesDatasetGenerator`
- Model hyperparameter configuration
- `TSForecasting` pipeline setup and fitting
- Results extraction and forecasting

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=Warning)

from tsforecasting import (
    TSForecasting,
    TimeSeriesDatasetGenerator,
    model_configurations,
)

---
## 1. Data Generation

The `TimeSeriesDatasetGenerator` class provides utilities for generating synthetic time series data with configurable patterns.

In [ ]:
# List available options
print("Available granularities:")
for granularity in TimeSeriesDatasetGenerator.list_available_granularities():
    print(f"  {granularity}")

print("\nAvailable patterns:")
for pattern in TimeSeriesDatasetGenerator.list_available_patterns():
    print(f"  {pattern}")

In [ ]:
# Generate synthetic data
data = TimeSeriesDatasetGenerator.generate(
    n_samples=500,
    granularity="1mo",
    patterns=["trend", "seasonal"],
    trend_strength=0.3,
    trend_type="linear",
    seasonality_period=12,
    seasonality_strength=1.5,
    noise_level=0.1,
    start_date="2000-01-01",
    base_value=100.0,
    random_state=42,
)

print(f"Generated data shape: {data.shape}")
data.head()

### Alternative: Quick Generators

```python
data = TimeSeriesDatasetGenerator.quick_monthly(n_samples=120)
data = TimeSeriesDatasetGenerator.quick_daily(n_samples=365)
data = TimeSeriesDatasetGenerator.quick_hourly(n_samples=720)
```

---
## 2. Model Hyperparameters

The `model_configurations()` function returns a dictionary of default hyperparameters for all available models.

In [ ]:
# Get default configurations
hparameters = model_configurations()

print("Default configurations:")
for model, params in hparameters.items():
    print(f"\n  {model}:")
    for key, value in params.items():
        print(f"    {key}: {value}")

In [ ]:
# Customize hyperparameters
hparameters["RandomForest"]["n_estimators"] = 50
hparameters["XGBoost"]["n_estimators"] = 50
hparameters["Catboost"]["iterations"] = 50
hparameters["KNN"]["n_neighbors"] = 5
hparameters["GBR"]["learning_rate"] = 0.05

print("Hyperparameters customized.")

---
## 3. Pipeline Configuration

Configure the `TSForecasting` pipeline with the desired parameters.

In [ ]:
# Available options reference
AVAILABLE_MODELS = [
    "RandomForest",
    "ExtraTrees",
    "GBR",
    "KNN",
    "GeneralizedLR",
    "XGBoost",
    "LightGBM",
    "Catboost",
    "AutoGluon",
]
AVAILABLE_METRICS = ["MAE", "MAPE", "MSE"]
AVAILABLE_GRANULARITIES = ["1m", "30m", "1h", "1d", "1wk", "1mo"]

print(f"Available models: {AVAILABLE_MODELS}")
print(f"Available metrics: {AVAILABLE_METRICS}")
print(f"Available granularities: {AVAILABLE_GRANULARITIES}")

In [ ]:
# Configure pipeline
tsf = TSForecasting(
    train_size=0.80,
    lags=12,
    horizon=6,
    sliding_size=6,
    models=[
        "RandomForest",
        "GBR",
        "KNN",
        "XGBoost",
    ],
    hparameters=hparameters,
    granularity="1mo",
    metric="MAE",
)

print("Pipeline configured:")
print(f"  train_size: {tsf.train_size}")
print(f"  lags: {tsf.lags}")
print(f"  horizon: {tsf.horizon}")
print(f"  sliding_size: {tsf.sliding_size}")
print(f"  metric: {tsf.metric}")

---
## 4. Fit and Evaluate

The `fit_forecast` method trains and evaluates all selected models using the expanding window approach.

In [ ]:
# Fit the pipeline (expanding window evaluation)
tsf.fit_forecast(dataset=data)

print("Pipeline fitted.")
print(f"Best model: {tsf.selected_model}")

---
## 5. Results Extraction

The `history()` method returns a `PerformanceHistory` dataclass containing detailed evaluation results.

In [ ]:
# Get history
history = tsf.history()

print(f"History type: {type(history).__name__}")

### 5.1 Predictions DataFrame

Raw forecasts per window with timestamps.

In [ ]:
predictions = history.predictions

print(f"Shape: {predictions.shape}")
predictions.head()

### 5.2 Performance Complete

Detailed metrics per window and horizon.

In [ ]:
performance_complete = history.performance_complete

print(f"Shape: {performance_complete.shape}")
performance_complete.head(10)

### 5.3 Performance by Horizon

Aggregated metrics across windows per horizon. Useful for understanding error growth over the forecast horizon.

In [ ]:
performance_horizon = history.performance_by_horizon

performance_horizon

### 5.4 Leaderboard

Model rankings by the selected metric.

In [ ]:
leaderboard = history.leaderboard

leaderboard

### 5.5 Selected Model

In [ ]:
print(f"Best Model: {history.selected_model}")

---
## 6. Generate Forecast

The `forecast` method generates future predictions using the best model with configurable prediction intervals.

**Interval Methods:**
- `ensemble`: Average of all methods (default, most robust)
- `quantile`: Empirical percentiles (captures asymmetry)
- `conformal`: Coverage guarantee (symmetric)
- `gaussian`: Parametric mean +/- z*std

In [ ]:
# Generate future forecast with prediction intervals
forecast = tsf.forecast(dataset=data, interval_method="ensemble")

print(f"Forecast ({tsf.horizon} steps ahead):")
forecast[["Date", "y"]]

In [ ]:
# Forecast with prediction intervals
forecast[["Date", "y", "y_lower_90", "y_upper_90"]]

---
## 7. Quick Reference

### Minimal Usage

```python
from tsforecasting import TSForecasting, TimeSeriesDatasetGenerator

data = TimeSeriesDatasetGenerator.quick_monthly(n_samples=200)
tsf = TSForecasting(lags=12, horizon=6, models=["RandomForest", "XGBoost"])
tsf.fit_forecast(data)
forecast = tsf.forecast()
```

### Full Configuration

```python
tsf = TSForecasting(
    train_size=0.80,
    lags=12,
    horizon=6,
    sliding_size=10,
    models=["RandomForest", "XGBoost", "LightGBM"],
    hparameters=model_configurations(),
    granularity="1mo",
    metric="MAE",
)
```

### Data Generation

```python
# Synthetic with patterns
data = TimeSeriesDatasetGenerator.generate(
    n_samples=300,
    granularity="1mo",
    patterns=["trend", "seasonal"],
)

# Quick generators
data = TimeSeriesDatasetGenerator.quick_monthly()
data = TimeSeriesDatasetGenerator.quick_daily()
data = TimeSeriesDatasetGenerator.quick_hourly()
```